<a href="https://colab.research.google.com/github/harneet862/ur2phd-assignment/blob/main/part4_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 4: analysis
Reads the prediction files and official scorer output already in this repository. No GPU needed.

In [33]:
!git clone -q https://github.com/harneet862/ur2phd-assignment.git
!git clone -q https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git

fatal: destination path 'ur2phd-assignment' already exists and is not an empty directory.
fatal: destination path 'SemEval-2027-Task-2-DiCo-NLI' already exists and is not an empty directory.


In [34]:
import json, pandas as pd

REPO = "/content/ur2phd-assignment"
TASK = "/content/SemEval-2027-Task-2-DiCo-NLI"
ORDER = ["EQUIVALENCE", "FORWARD_ENTAILMENT", "BACKWARD_ENTAILMENT", "NEGATIVE_OTHER"]

SYSTEMS = {
    "Majority":        f"{REPO}/results-part0",
    "DeBERTa seed13":  f"{REPO}/results-part2/seed13",
    "DeBERTa seed2026":f"{REPO}/results-part2/seed2026",
    "LLM zero-shot":   f"{REPO}/results-part3/zeroshot",
    "LLM few-shot":    f"{REPO}/results-part3/fewshot",
}
scores = {name: json.load(open(f"{path}/scores.json")) for name, path in SYSTEMS.items()}

# gold file: text, gold label and reverse_pair_id for every dev item
ref = pd.read_csv(f"{TASK}/final_data/dev/dico_nli_dev_track1_reference.csv")

## Q1: comparison of all systems, and with the pilot results in the task README

In [35]:
table = pd.DataFrame({name: {"weighted_f1": s["weighted_f1"], "soft_cons": s["soft_cons"], "hard_cons": s["hard_cons"]}
                      for name, s in scores.items()}).T
print(table.round(3))

# pilot results copied from the task README ("Pilot Results", positive combined track)
pilot = pd.DataFrame({
    "Pilot DeBERTa-v3-base (encoder)": [0.75, 0.84, 0.79],
    "Pilot OPT-125M (decoder)":        [0.61, 0.65, 0.58],
    "Pilot BART-large (enc-dec)":      [0.68, 0.80, 0.72],
}, index=["weighted_f1", "soft_cons", "hard_cons"]).T
print()
print(pilot)

                  weighted_f1  soft_cons  hard_cons
Majority                0.129      0.000      0.000
DeBERTa seed13          0.823      0.884      0.834
DeBERTa seed2026        0.815      0.895      0.834
LLM zero-shot           0.234      0.260      0.123
LLM few-shot            0.263      0.343      0.148

                                 weighted_f1  soft_cons  hard_cons
Pilot DeBERTa-v3-base (encoder)         0.75       0.84       0.79
Pilot OPT-125M (decoder)                0.61       0.65       0.58
Pilot BART-large (enc-dec)              0.68       0.80       0.72


## Q2: confusion matrices from the scorer (rows = gold, columns = predicted)

In [36]:
for name in ["DeBERTa seed13", "DeBERTa seed2026", "LLM zero-shot", "LLM few-shot"]:
    cm = pd.DataFrame(scores[name]["classification"]["confusion_matrix"]).T.loc[ORDER, ORDER]
    f1 = {lab: round(v["f1"], 3) for lab, v in scores[name]["classification"]["per_label"].items()}
    print(f"{name}:")
    print(cm.to_string())
    print("per-label F1:", f1, "\n")

DeBERTa seed13:
                     EQUIVALENCE  FORWARD_ENTAILMENT  BACKWARD_ENTAILMENT  NEGATIVE_OTHER
EQUIVALENCE                  139                   6                    5              24
FORWARD_ENTAILMENT            11                 166                    2              11
BACKWARD_ENTAILMENT            9                   2                  164              15
NEGATIVE_OTHER                15                   9                   10              72
per-label F1: {'BACKWARD_ENTAILMENT': 0.884, 'EQUIVALENCE': 0.799, 'FORWARD_ENTAILMENT': 0.89, 'NEGATIVE_OTHER': 0.632} 

DeBERTa seed2026:
                     EQUIVALENCE  FORWARD_ENTAILMENT  BACKWARD_ENTAILMENT  NEGATIVE_OTHER
EQUIVALENCE                  138                   8                    6              22
FORWARD_ENTAILMENT            13                 166                    1              10
BACKWARD_ENTAILMENT           10                   2                  165              13
NEGATIVE_OTHER                15  

## Q3: errors of the best system (DeBERTa, seed 13)
Seed 13 has the higher weighted F1 of the two seeds.

In [37]:
p13 = pd.read_csv(f"{REPO}/results-part2/seed13/predictions_deberta_seed13.csv").rename(columns={"label": "pred_seed13"})
p26 = pd.read_csv(f"{REPO}/results-part2/seed2026/predictions_deberta_seed2026.csv").rename(columns={"label": "pred_seed2026"})
dev = ref.merge(p13, on="instance_id").merge(p26, on="instance_id")

# rows where the seed 13 is wrong
errors = dev[dev["label"] != dev["pred_seed13"]]
print("Error rows seen in seed 13:", len(errors), "of", len(dev))
# rows where the seed 2026 is wrong
print("Error rows seen in both seed 13 and seed 2026:", (errors["pred_seed2026"] != errors["label"]).sum())
# find how many of each combinations were seen
print(pd.crosstab(errors["label"], errors["pred_seed13"]))

Error rows seen in seed 13: 119 of 660
Error rows seen in both seed 13 and seed 2026: 103
pred_seed13          BACKWARD_ENTAILMENT  EQUIVALENCE  FORWARD_ENTAILMENT  \
label                                                                       
BACKWARD_ENTAILMENT                    0            9                   2   
EQUIVALENCE                            5            0                   6   
FORWARD_ENTAILMENT                     2           11                   0   
NEGATIVE_OTHER                        10           15                   9   

pred_seed13          NEGATIVE_OTHER  
label                                
BACKWARD_ENTAILMENT              15  
EQUIVALENCE                      24  
FORWARD_ENTAILMENT               11  
NEGATIVE_OTHER                    0  


In [38]:
print(errors[["instance_id","text1", "text2", "label", "pred_seed13"]].to_string())

                               instance_id                                       text1                                       text2                label          pred_seed13
8     dico_phrasis_0000018__en-en__flipped                         of all Rage rallies                              march of anger   FORWARD_ENTAILMENT          EQUIVALENCE
9    dico_phrasis_0000018__en-en__original                              march of anger                         of all Rage rallies  BACKWARD_ENTAILMENT       NEGATIVE_OTHER
16    dico_phrasis_0000024__en-en__flipped                               in destroying                                     destroy   FORWARD_ENTAILMENT          EQUIVALENCE
26    dico_phrasis_0000059__en-en__flipped      for costa concordia shipwreck disaster               over Costa Concordia disaster   FORWARD_ENTAILMENT       NEGATIVE_OTHER
27   dico_phrasis_0000059__en-en__original               over Costa Concordia disaster      for costa concordia shipwreck disaster  BAC

In [39]:
# the 10 errors discussed in the report, picked by hand from the full error list above
CHOSEN_ERRORS = [
    "dico_phrasis_0001781__en-en__original",   # at the train station -> at the station
    "dico_phrasis_0001926__en-en__original",   # a video game -> game
    "dico_phrasis_0000198__en-en__flipped",    # a chocolate bar -> a candy bar
    "dico_phrasis_0000711__en-en__original",   # to pay off -> of bribing
    "dico_phrasis_0001103__en-en__flipped",    # A ginger and white cat -> A orange and white cat
    "dico_phrasis_0000895__en-en__original",   # in Texas fertilizer plant blast -> of Texas fertiliser plant blast
    "dico_phrasis_0002844__en-en__original",   # Northwest China -> in China
    "dico_phrasis_0002560__en-en__original",   # Two young children -> Two small children
    "dico_phrasis_0001551__en-en__flipped",    # Saudi Women -> Saudis
    "dico_phrasis_0001659__en-en__flipped",    # $ 25bn oil link -> $ 30 billion oil link
]
print(dev.set_index("instance_id").loc[CHOSEN_ERRORS, ["text1", "text2", "label", "pred_seed13", "pred_seed2026"]].to_string())

                                                                 text1                            text2                label         pred_seed13        pred_seed2026
instance_id                                                                                                                                                          
dico_phrasis_0001781__en-en__original             at the train station                   at the station   FORWARD_ENTAILMENT         EQUIVALENCE          EQUIVALENCE
dico_phrasis_0001926__en-en__original                     a video game                             game   FORWARD_ENTAILMENT         EQUIVALENCE          EQUIVALENCE
dico_phrasis_0000198__en-en__flipped                   a chocolate bar                      a candy bar   FORWARD_ENTAILMENT         EQUIVALENCE          EQUIVALENCE
dico_phrasis_0000711__en-en__original                       to pay off                       of bribing          EQUIVALENCE      NEGATIVE_OTHER       NEGATIVE_OTHER
dico

Error analysis for DeBERTa, seed 13:

The model makes 119 errors on dev, and 103 of them are also made by seed 2026, so the errors are systematic rather than random. Very few confuse FORWARD with BACKWARD (4 of 119). The model handles direction well, and most errors are about the boundaries between labels. I observed the four patterns below:

1. Word overlap seems to drive the decision. When one phrase is only slightly more specific than the other, the model predicts EQUIVALENCE instead of entailment: "at the train station" → "at the station" and "a video game" → "game" are both gold FORWARD_ENTAILMENT. In the other direction, when two phrases mean the same thing but share few words, the model predicts NEGATIVE_OTHER: "of bribing" / "to pay off" and "A ginger and white cat" / "A orange and white cat" are gold EQUIVALENCE. EQUIVALENCE predicted as NEGATIVE_OTHER is the most common error type (24 cases).

2. Relations that need world knowledge are missed. "in a grassy green field" → "in a yard" and "of some runway" → "from the airport" are gold FORWARD_ENTAILMENT, but the model predicts NEGATIVE_OTHER. The link is not visible in the words, and the model does not seem to know that a runway is part of an airport.

3. Small words decide NEGATIVE_OTHER, and the model ignores them. "Northwest China" → "in China" and "in a top hat" / "with a top hat" are gold NEGATIVE_OTHER, but the model predicts a relation because almost all words are shared. Some of these cases are hard even for a human, which may explain why NEGATIVE_OTHER is the weakest label (F1 0.63).

4. An extra word or a different word order is read as extra meaning. "May Housing" / "Housing in May" is gold EQUIVALENCE, but the model predicts entailment. It still gives compatible labels in the two directions (FORWARD one way, BACKWARD the other), so this error is consistent but wrong.

One case may be a gold label issue: "Saudi Women" → "Saudis" is gold BACKWARD_ENTAILMENT, but by the label definitions, Saudi women are Saudis, so the model's FORWARD_ENTAILMENT looks correct.

## Q4: pairs that fail SoftCons (DeBERTa, seed 13)
The scorer lists every failed pair in `pair_errors`. NEGATIVE_OTHER has no reverse in the scorer's reversal map, so predicting NEGATIVE_OTHER for either member of a reversible pair always counts as inconsistent.

In [40]:
text = ref.set_index("instance_id")
pairs = pd.DataFrame(scores["DeBERTa seed13"]["consistency"]["pair_errors"])

soft_fail = pairs[pairs["soft_consistent"] == "false"].copy()
soft_fail["text1"] = soft_fail["first_id"].map(text["text1"])
soft_fail["text2"] = soft_fail["first_id"].map(text["text2"])

cols = ["text1", "text2", "first_gold", "first_prediction", "second_prediction"]
print("SoftCons failures:", len(soft_fail))
print(soft_fail[cols].to_string(index=False))

SoftCons failures: 32
                                  text1                                      text2          first_gold   first_prediction   second_prediction
                    of all Rage rallies                             march of anger  FORWARD_ENTAILMENT        EQUIVALENCE      NEGATIVE_OTHER
                          in destroying                                    destroy  FORWARD_ENTAILMENT        EQUIVALENCE BACKWARD_ENTAILMENT
 for costa concordia shipwreck disaster              over Costa Concordia disaster  FORWARD_ENTAILMENT     NEGATIVE_OTHER      NEGATIVE_OTHER
     in an overgrown green grassy field                        through green grass  FORWARD_ENTAILMENT     NEGATIVE_OTHER      NEGATIVE_OTHER
                     in front of a pool                                  at a pool  FORWARD_ENTAILMENT FORWARD_ENTAILMENT      NEGATIVE_OTHER
                in a grassy green field                                  in a yard  FORWARD_ENTAILMENT     NEGATIVE_OTHER     

In [41]:
# the 5 inconsistent pairs discussed in the report (first_id of each pair)
CHOSEN_PAIRS = [
    "dico_phrasis_0000024__en-en__flipped",    # in destroying / destroy
    "dico_phrasis_0000141__en-en__flipped",    # in front of a pool / at a pool
    "dico_phrasis_0000375__en-en__flipped",    # to deliver statement / to speak
    "dico_phrasis_0000936__en-en__flipped",    # fresh presidential election / fresh election for president
    "dico_phrasis_0001659__en-en__flipped",    # $ 25bn oil link / $ 30 billion oil link
]
print(soft_fail.set_index("first_id").loc[CHOSEN_PAIRS, cols].to_string())

                                                            text1                         text2           first_gold    first_prediction    second_prediction
first_id                                                                                                                                                     
dico_phrasis_0000024__en-en__flipped                in destroying                       destroy   FORWARD_ENTAILMENT         EQUIVALENCE  BACKWARD_ENTAILMENT
dico_phrasis_0000141__en-en__flipped           in front of a pool                     at a pool   FORWARD_ENTAILMENT  FORWARD_ENTAILMENT       NEGATIVE_OTHER
dico_phrasis_0000375__en-en__flipped         to deliver statement                      to speak   FORWARD_ENTAILMENT  FORWARD_ENTAILMENT          EQUIVALENCE
dico_phrasis_0000936__en-en__flipped  fresh presidential election  fresh election for president          EQUIVALENCE         EQUIVALENCE   FORWARD_ENTAILMENT
dico_phrasis_0001659__en-en__flipped              $ 

In all five pairs, the model gets one direction right and the other wrong. For example, "to deliver statement" → "to speak" is correctly predicted as FORWARD_ENTAILMENT, but the reversed pair is predicted as EQUIVALENCE instead of BACKWARD_ENTAILMENT. The wrong direction always moves to a neighbouring label (entailment becomes EQUIVALENCE or NEGATIVE_OTHER), which shows these are borderline pairs where the two phrases are close in meaning.

The model treats each direction as a separate input, and in training, we dont make the two predictions agree. For borderline pairs, swapping the order of the phrases is enough to push one direction across a decision boundary.

Most SoftCons failures are actually a different kind of error: 21 of the 32 failed pairs are predicted NEGATIVE_OTHER in both directions. The model is symmetric there, but it misses that the phrases are related, and the scorer counts NEGATIVE_OTHER on a reversible pair as inconsistent. Only 11 pairs have truly incompatible labels in the two directions.